# CS Domain Expert — Synthetic Fine-Tuning Data Generator

## Pipeline Overview

This notebook generates **high-quality, subject-aware synthetic fine-tuning datasets** for training per-subject LoRA/QLoRA adapters on a base SLM, plus a **router training dataset** for prompt routing.

### Architecture
```
PDFs (per subject)
    ├── Text Extraction  (PyMuPDF)
    ├── Image Extraction (PyMuPDF)
    │       └── Vision Captioning  (VLM)
    │               └── Image Q&A Generation (VLM)
    ├── Text Cleaning + Chunking
    ├── Embedding  (Embedding Model)
    └── VectorDB (ChromaDB)
            └── RAG-Augmented Synthetic Data (LLM)
                    ├── Conceptual Q&A
                    ├── Multiple Choice (MCQ)
                    ├── Code / Algorithm Problems
                    ├── True/False + Explanation
                    └── Scenario / Application
```

### Outputs
- `outputs/subjects/<SUBJECT>_finetune.jsonl`  — per-subject dataset
- `outputs/combined_finetune.jsonl`            — merged dataset
- `outputs/router_training.jsonl`              — subject router training data
- `outputs/pipeline_stats.json`                — generation statistics

### Naming Convention for Books
PDFs must be named `<SUBJECT_CODE> - <N>.pdf` (e.g. `CAO - 1.pdf`, `DSA - 2.pdf`).
The notebook auto-discovers all subject codes and groups books accordingly.

---
**Supported question types per subject:**

| Type | CAO | CN | DSA | OS |
|---|---|---|---|---|
| Conceptual Q&A | ✓ | ✓ | ✓ | ✓ |
| MCQ | ✓ | ✓ | ✓ | ✓ |
| Code / Algorithm | ✓ | ✓ | ✓✓ | ✓ |
| True/False | ✓ | ✓ | ✓ | ✓ |
| Diagram Q&A (Vision) | ✓✓ | ✓ | ✓ | ✓✓ |
| Scenario | ✓ | ✓✓ | ✓ | ✓✓ |

## Cell 1 — Install Dependencies

In [ ]:
# Uncomment on a fresh environment
# %pip install -q openai chromadb pillow tqdm python-dotenv pymupdf datasets

## Cell 2 — Imports, Logging, and Environment

In [1]:
import base64
import hashlib
import json
import logging
import os
import re
import time
from collections import defaultdict
from datetime import datetime, timezone
from io import BytesIO
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import chromadb
import fitz
from dotenv import load_dotenv
from openai import OpenAI
from PIL import Image
from tqdm.auto import tqdm

load_dotenv()

# ---------------------------------------------------------------------------
# Logging
# ---------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler("pipeline.log", encoding="utf-8"),
    ],
)
logger = logging.getLogger("cs-synth-pipeline")

logger.info("Environment loaded. Libraries imported.")

2026-03-24 20:14:12,530 | INFO | Environment loaded. Libraries imported.


## Cell 3 — API Clients and Directory Layout

In [2]:
# ---------------------------------------------------------------------------
# API Configuration  (set via .env or shell environment)
# ---------------------------------------------------------------------------
VISION_BASE_URL   = os.getenv("VISION_BASE_URL",   "h")
VISION_API_KEY    = os.getenv("VISION_API_KEY",    "")
VISION_MODEL      = os.getenv("VISION_MODEL",      "Qwen/Qwen3-VL-8B-Instruct")

TEXT_BASE_URL     = os.getenv("TEXT_BASE_URL",     "")
TEXT_API_KEY      = os.getenv("TEXT_API_KEY",      "")
SYNTHETIC_MODEL   = os.getenv("SYNTHETIC_MODEL",   "openai/gpt-oss-20b")

EMBEDDING_BASE_URL = os.getenv("EMBEDDING_BASE_URL", "")
EMBEDDING_API_KEY  = os.getenv("EMBEDDING_API_KEY",  "")
EMBEDDING_MODEL    = os.getenv("EMBEDDING_MODEL",    "Qwen/Qwen3-Embedding-0.6B")

vision_client    = OpenAI(base_url=VISION_BASE_URL,    api_key=VISION_API_KEY    or "x")
text_client      = OpenAI(base_url=TEXT_BASE_URL,      api_key=TEXT_API_KEY      or "x")
embedding_client = OpenAI(base_url=EMBEDDING_BASE_URL, api_key=EMBEDDING_API_KEY or "x")

# ---------------------------------------------------------------------------
# Directory Layout
# ---------------------------------------------------------------------------
ROOT          = Path.cwd().resolve()
BOOKS_DIR     = ROOT.parents[1] / "datasets" / "training" / "Books"
IMAGE_DIR     = ROOT / "images" / "books"
VECTORDB_DIR  = ROOT / "vectordb"
CHECKPOINT_DIR = ROOT / "checkpoints"
OUTPUT_DIR    = ROOT / "outputs"
SUBJECT_OUT   = OUTPUT_DIR / "subjects"

for d in [IMAGE_DIR, VECTORDB_DIR, CHECKPOINT_DIR, OUTPUT_DIR, SUBJECT_OUT]:
    d.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------------
# Generation hyper-parameters  (tune these per your API budget)
# ---------------------------------------------------------------------------
MAX_CHUNKS_PER_BOOK  = 12     # text chunks used for generation per book
PAIRS_PER_CHUNK      = 3      # Q&A pairs generated per chunk per question type
CHUNK_CHARS          = 1800   # characters per text chunk
CHUNK_OVERLAP        = 250    # overlap between chunks
EMBED_BATCH          = 32     # sentences per embedding API call
UPSERT_BATCH         = 64     # records per ChromaDB upsert
API_RETRY_ATTEMPTS   = 4      # retries on transient API errors
API_RETRY_DELAY      = 2.0    # seconds between retries (doubles each time)
MIN_IMAGE_WIDTH      = int(os.getenv("MIN_IMAGE_WIDTH",  "250"))
MIN_IMAGE_HEIGHT     = int(os.getenv("MIN_IMAGE_HEIGHT", "250"))

print("BOOKS_DIR  :", BOOKS_DIR)
print("IMAGE_DIR  :", IMAGE_DIR)
print("OUTPUT_DIR :", OUTPUT_DIR)

BOOKS_DIR  : /media/sasank-v/New Volume/Studies/College/Interships/LLM-Finetuning/datasets/training/Books
IMAGE_DIR  : /media/sasank-v/New Volume/Studies/College/Interships/LLM-Finetuning/notebooks/preprocessing/images/books
OUTPUT_DIR : /media/sasank-v/New Volume/Studies/College/Interships/LLM-Finetuning/notebooks/preprocessing/outputs


## Cell 4 — Subject Registry

Maps short subject codes (inferred from PDF filenames) to rich metadata used for prompt construction and dataset labelling.

In [3]:
# ---------------------------------------------------------------------------
# Subject Registry
# Keys must match the prefix used in PDF filenames (e.g. "CAO" in "CAO - 1.pdf")
# ---------------------------------------------------------------------------
SUBJECT_REGISTRY: Dict[str, Dict[str, Any]] = {
    "CAO": {
        "full_name": "Computer Architecture and Organization",
        "description": (
            "Covers processor design, instruction set architectures (ISA), "
            "pipelining, memory hierarchy, cache, I/O systems, and parallel architectures."
        ),
        "key_topics": [
            "ISA", "RISC vs CISC", "pipelining", "hazards", "cache coherence",
            "memory hierarchy", "virtual memory", "I/O", "multiprocessors",
            "ALU design", "control unit", "branch prediction",
        ],
        # Weight of each question type — controls how many chunks go to each type
        "qtype_weights": {
            "conceptual":  4,
            "mcq":         3,
            "code":        2,
            "truefalse":   2,
            "diagram":     4,   # many diagrams in CAO books
            "scenario":    1,
        },
        "code_language": "Assembly / C",
    },
    "CN": {
        "full_name": "Computer Networks",
        "description": (
            "Covers OSI/TCP-IP models, protocols (HTTP, TCP, UDP, IP, DNS, DHCP), "
            "routing algorithms, network security, wireless networks, and socket programming."
        ),
        "key_topics": [
            "OSI model", "TCP/IP", "IP addressing", "subnetting", "routing",
            "TCP vs UDP", "DNS", "HTTP", "TLS", "NAT", "VLAN",
            "congestion control", "socket programming", "CDN",
        ],
        "qtype_weights": {
            "conceptual":  3,
            "mcq":         3,
            "code":        2,
            "truefalse":   2,
            "diagram":     2,
            "scenario":    4,   # CN has lots of scenario-based Qs
        },
        "code_language": "Python / C sockets",
    },
    "DSA": {
        "full_name": "Data Structures and Algorithms",
        "description": (
            "Covers arrays, linked lists, stacks, queues, trees, graphs, hashing, "
            "sorting, searching, dynamic programming, greedy algorithms, and complexity analysis."
        ),
        "key_topics": [
            "arrays", "linked lists", "stacks", "queues", "binary trees",
            "BST", "heaps", "graphs", "BFS", "DFS", "Dijkstra",
            "sorting algorithms", "dynamic programming", "Big-O notation",
            "hashing", "tries", "segment trees",
        ],
        "qtype_weights": {
            "conceptual":  2,
            "mcq":         2,
            "code":        6,   # DSA is code-heavy
            "truefalse":   2,
            "diagram":     2,
            "scenario":    2,
        },
        "code_language": "Python / C++",
    },
    "OS": {
        "full_name": "Operating Systems",
        "description": (
            "Covers process management, scheduling, synchronization, deadlocks, "
            "memory management, virtual memory, file systems, I/O, and security."
        ),
        "key_topics": [
            "processes", "threads", "CPU scheduling", "synchronization",
            "mutex", "semaphores", "deadlocks", "paging", "segmentation",
            "virtual memory", "page replacement", "file systems",
            "I/O management", "system calls", "protection",
        ],
        "qtype_weights": {
            "conceptual":  4,
            "mcq":         3,
            "code":        2,
            "truefalse":   2,
            "diagram":     3,
            "scenario":    4,   # OS is rich in scenario Qs
        },
        "code_language": "C / Python",
    },
}


def get_subject_meta(subject_code: str) -> Dict[str, Any]:
    """Return registry entry, or a sensible default for unknown subjects."""
    return SUBJECT_REGISTRY.get(
        subject_code.upper(),
        {
            "full_name": subject_code,
            "description": f"Computer Science subject: {subject_code}",
            "key_topics": [],
            "qtype_weights": {
                "conceptual": 3, "mcq": 3, "code": 2,
                "truefalse": 2, "diagram": 2, "scenario": 2,
            },
            "code_language": "Python / C",
        },
    )


print("Registered subjects:", list(SUBJECT_REGISTRY.keys()))

Registered subjects: ['CAO', 'CN', 'DSA', 'OS']


## Cell 5 — Utility Functions (Retry, JSON, Hashing)

In [4]:
def with_retry(fn, attempts: int = API_RETRY_ATTEMPTS, delay: float = API_RETRY_DELAY):
    """Call fn(); retry with exponential back-off on any exception."""
    for attempt in range(1, attempts + 1):
        try:
            return fn()
        except Exception as exc:
            if attempt == attempts:
                raise
            wait = delay * (2 ** (attempt - 1))
            logger.warning("Attempt %d/%d failed (%s). Retrying in %.1fs…", attempt, attempts, exc, wait)
            time.sleep(wait)


def load_json(path: Path, default: Any = None) -> Any:
    if path.exists():
        try:
            return json.loads(path.read_text(encoding="utf-8"))
        except Exception:
            pass
    return default if default is not None else {}


def dump_json(path: Path, payload: Any) -> None:
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")


def stable_hash(text: str) -> str:
    """Short deterministic identifier for deduplication."""
    return hashlib.sha256(text.encode()).hexdigest()[:16]


def extract_json_array(raw: str) -> List[Dict]:
    """Robustly parse a JSON array from a model response that may have markdown fences."""
    # Strip markdown code fences
    cleaned = re.sub(r"```(?:json)?\s*", "", raw).strip().rstrip("`").strip()
    # Try direct parse
    try:
        result = json.loads(cleaned)
        if isinstance(result, list):
            return result
        if isinstance(result, dict):
            # Some models wrap the array in {"data": [...]}
            for v in result.values():
                if isinstance(v, list):
                    return v
    except json.JSONDecodeError:
        pass
    # Fallback: find first [...] block
    m = re.search(r"\[.*\]", cleaned, re.DOTALL)
    if m:
        try:
            return json.loads(m.group(0))
        except json.JSONDecodeError:
            pass
    logger.warning("Failed to parse JSON array from model output (len=%d)", len(raw))
    return []


def chunked(seq: List, size: int):
    for i in range(0, len(seq), size):
        yield seq[i : i + size]


print("Utilities defined.")

Utilities defined.


## Cell 6 — Discover Books by Subject

In [5]:
def discover_subjects(books_dir: Path) -> Dict[str, List[Path]]:
    """
    Scan books_dir for PDFs with the naming pattern `<CODE> - <N>.pdf`.
    Returns a dict mapping subject code → sorted list of PDF paths.
    """
    subject_map: Dict[str, List[Path]] = defaultdict(list)
    pattern = re.compile(r"^([A-Za-z0-9]+)\s*-\s*\d+\.pdf$", re.IGNORECASE)
    for pdf in sorted(books_dir.glob("*.pdf")):
        m = pattern.match(pdf.name)
        if m:
            code = m.group(1).upper()
            subject_map[code].append(pdf)
        else:
            logger.warning("Skipping non-standard filename: %s", pdf.name)
    return dict(subject_map)


if not BOOKS_DIR.exists():
    raise FileNotFoundError(f"Books directory not found: {BOOKS_DIR}")

subject_books = discover_subjects(BOOKS_DIR)

print(f"Discovered {len(subject_books)} subject(s):")
for code, pdfs in subject_books.items():
    meta = get_subject_meta(code)
    print(f"  {code:6s} → {meta['full_name']}  ({len(pdfs)} book(s))")
    for p in pdfs:
        print(f"           {p.name}")

Discovered 4 subject(s):
  CAO    → Computer Architecture and Organization  (2 book(s))
           CAO - 1.pdf
           CAO - 2.pdf
  CN     → Computer Networks  (3 book(s))
           CN - 1.pdf
           CN - 2.pdf
           CN - 3.pdf
  DSA    → Data Structures and Algorithms  (4 book(s))
           DSA - 1.pdf
           DSA - 2.pdf
           DSA - 3.pdf
           DSA - 4.pdf
  OS     → Operating Systems  (3 book(s))
           OS - 1.pdf
           OS - 2.pdf
           OS - 3.pdf


## Cell 7 — PDF Text Extraction

In [6]:
def extract_text_from_pdf(pdf_path: Path) -> str:
    """
    Extract raw text from every page using three fallback strategies:
    1) Standard text mode  2) Blocks mode  3) Dict/spans mode
    """
    fitz.TOOLS.mupdf_display_errors(False)
    doc = fitz.open(pdf_path)
    pages_text: List[str] = []
    total = len(doc)

    for page_num, page in enumerate(doc):
        txt = page.get_text("text")
        if not txt.strip():
            blocks = page.get_text("blocks")
            txt = "\n".join(b[4] for b in blocks if len(b) > 4)
        if not txt.strip():
            d = page.get_text("dict")
            parts = []
            for blk in d.get("blocks", []):
                if blk.get("type") == 0:
                    for line in blk.get("lines", []):
                        for span in line.get("spans", []):
                            parts.append(span.get("text", ""))
            txt = " ".join(parts)
        pages_text.append(txt)
        if (page_num + 1) % 200 == 0:
            logger.info("  %s: %d/%d pages processed", pdf_path.name, page_num + 1, total)

    doc.close()
    combined = "\n".join(pages_text)
    logger.info("✓ %s: %d chars from %d pages", pdf_path.name, len(combined), total)
    return combined


def clean_book_text(raw: str) -> str:
    """Remove headers/footers, fix broken lines, normalise whitespace."""
    lines = raw.split("\n")
    skip_headings = {"index", "acknowledgment", "acknowledgements", "references", "bibliography"}
    cleaned, skip = [], False
    for line in lines:
        stripped = line.strip()
        if stripped.lower() in skip_headings:
            skip = True
            continue
        if re.fullmatch(r"\d+", stripped):
            continue
        if not skip:
            cleaned.append(line)
    text = "\n".join(cleaned)
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def chunk_text(text: str, chunk_chars: int = CHUNK_CHARS, overlap: int = CHUNK_OVERLAP) -> List[str]:
    chunks, start, n = [], 0, len(text)
    while start < n:
        end = min(n, start + chunk_chars)
        chunks.append(text[start:end])
        if end >= n:
            break
        start = max(0, end - overlap)
    return chunks


# ---------------------------------------------------------------------------
# Extract text for ALL subjects, checkpoint to disk
# ---------------------------------------------------------------------------
TEXT_CHECKPOINT = CHECKPOINT_DIR / "pdf_texts.json"
pdf_texts: Dict[str, str] = load_json(TEXT_CHECKPOINT, default={})
newly_extracted = 0

for code, pdfs in subject_books.items():
    for pdf in tqdm(pdfs, desc=f"[{code}] Extracting text"):
        if pdf.name in pdf_texts:
            continue
        try:
            raw = extract_text_from_pdf(pdf)
            pdf_texts[pdf.name] = raw
            newly_extracted += 1
            dump_json(TEXT_CHECKPOINT, pdf_texts)
        except Exception as exc:
            logger.error("Text extraction failed for %s: %s", pdf.name, exc)

print(f"\nTotal PDFs with extracted text : {len(pdf_texts)}  (newly extracted: {newly_extracted})")

# Build cleaned text map
cleaned_texts: Dict[str, str] = {name: clean_book_text(raw) for name, raw in pdf_texts.items()}

# Build subject → merged text map (all books concatenated)
subject_text: Dict[str, str] = {}
for code, pdfs in subject_books.items():
    parts = [cleaned_texts.get(p.name, "") for p in pdfs]
    subject_text[code] = "\n\n".join(filter(None, parts))
    print(f"  {code}: {len(subject_text[code]):,} chars (merged)")

[CAO] Extracting text:   0%|          | 0/2 [00:00<?, ?it/s]

2026-03-24 20:15:29,509 | INFO |   CAO - 1.pdf: 200/1137 pages processed
2026-03-24 20:15:30,095 | INFO |   CAO - 1.pdf: 400/1137 pages processed
2026-03-24 20:15:30,477 | INFO |   CAO - 1.pdf: 600/1137 pages processed
2026-03-24 20:15:30,873 | INFO |   CAO - 1.pdf: 800/1137 pages processed
2026-03-24 20:15:31,202 | INFO |   CAO - 1.pdf: 1000/1137 pages processed
2026-03-24 20:15:31,437 | INFO | ✓ CAO - 1.pdf: 2611231 chars from 1137 pages
2026-03-24 20:15:31,689 | INFO |   CAO - 2.pdf: 200/881 pages processed
2026-03-24 20:15:31,884 | INFO |   CAO - 2.pdf: 400/881 pages processed
2026-03-24 20:15:32,078 | INFO |   CAO - 2.pdf: 600/881 pages processed
2026-03-24 20:15:32,275 | INFO |   CAO - 2.pdf: 800/881 pages processed
2026-03-24 20:15:32,367 | INFO | ✓ CAO - 2.pdf: 1919466 chars from 881 pages


[CN] Extracting text:   0%|          | 0/3 [00:00<?, ?it/s]

2026-03-24 20:15:33,583 | INFO |   CN - 1.pdf: 200/1269 pages processed
2026-03-24 20:15:34,343 | INFO |   CN - 1.pdf: 400/1269 pages processed
2026-03-24 20:15:35,826 | INFO |   CN - 1.pdf: 600/1269 pages processed
2026-03-24 20:15:36,521 | INFO |   CN - 1.pdf: 800/1269 pages processed
2026-03-24 20:15:37,372 | INFO |   CN - 1.pdf: 1000/1269 pages processed
2026-03-24 20:15:38,182 | INFO |   CN - 1.pdf: 1200/1269 pages processed
2026-03-24 20:15:38,385 | INFO | ✓ CN - 1.pdf: 2706422 chars from 1269 pages
2026-03-24 20:15:38,718 | INFO |   CN - 2.pdf: 200/889 pages processed
2026-03-24 20:15:38,918 | INFO |   CN - 2.pdf: 400/889 pages processed
2026-03-24 20:15:39,140 | INFO |   CN - 2.pdf: 600/889 pages processed
2026-03-24 20:15:39,342 | INFO |   CN - 2.pdf: 800/889 pages processed
2026-03-24 20:15:39,456 | INFO | ✓ CN - 2.pdf: 2105879 chars from 889 pages
2026-03-24 20:15:40,050 | INFO |   CN - 3.pdf: 200/915 pages processed
2026-03-24 20:15:40,483 | INFO |   CN - 3.pdf: 400/915 pag

[DSA] Extracting text:   0%|          | 0/4 [00:00<?, ?it/s]

2026-03-24 20:15:41,854 | INFO |   DSA - 1.pdf: 200/654 pages processed
2026-03-24 20:15:42,044 | INFO |   DSA - 1.pdf: 400/654 pages processed
2026-03-24 20:15:42,222 | INFO |   DSA - 1.pdf: 600/654 pages processed
2026-03-24 20:15:42,306 | INFO | ✓ DSA - 1.pdf: 1318958 chars from 654 pages
2026-03-24 20:15:42,593 | INFO |   DSA - 2.pdf: 200/620 pages processed
2026-03-24 20:15:42,680 | INFO |   DSA - 2.pdf: 400/620 pages processed
2026-03-24 20:15:42,770 | INFO |   DSA - 2.pdf: 600/620 pages processed
2026-03-24 20:15:42,781 | INFO | ✓ DSA - 2.pdf: 983409 chars from 620 pages
2026-03-24 20:15:43,341 | INFO |   DSA - 3.pdf: 200/605 pages processed
2026-03-24 20:15:43,604 | INFO |   DSA - 3.pdf: 400/605 pages processed
2026-03-24 20:15:43,861 | INFO |   DSA - 3.pdf: 600/605 pages processed
2026-03-24 20:15:43,899 | INFO | ✓ DSA - 3.pdf: 1006118 chars from 605 pages
2026-03-24 20:15:44,249 | INFO |   DSA - 4.pdf: 200/1313 pages processed
2026-03-24 20:15:44,413 | INFO |   DSA - 4.pdf: 4

[OS] Extracting text:   0%|          | 0/3 [00:00<?, ?it/s]

2026-03-24 20:15:45,936 | INFO |   OS - 1.pdf: 200/1278 pages processed
2026-03-24 20:15:46,134 | INFO |   OS - 1.pdf: 400/1278 pages processed
2026-03-24 20:15:46,366 | INFO |   OS - 1.pdf: 600/1278 pages processed
2026-03-24 20:15:46,601 | INFO |   OS - 1.pdf: 800/1278 pages processed
2026-03-24 20:15:46,826 | INFO |   OS - 1.pdf: 1000/1278 pages processed
2026-03-24 20:15:47,045 | INFO |   OS - 1.pdf: 1200/1278 pages processed
2026-03-24 20:15:47,145 | INFO | ✓ OS - 1.pdf: 3188670 chars from 1278 pages
2026-03-24 20:15:47,678 | INFO |   OS - 2.pdf: 200/1137 pages processed
2026-03-24 20:15:47,814 | INFO |   OS - 2.pdf: 400/1137 pages processed
2026-03-24 20:15:47,968 | INFO |   OS - 2.pdf: 600/1137 pages processed
2026-03-24 20:15:48,119 | INFO |   OS - 2.pdf: 800/1137 pages processed
2026-03-24 20:15:48,263 | INFO |   OS - 2.pdf: 1000/1137 pages processed
2026-03-24 20:15:48,404 | INFO | ✓ OS - 2.pdf: 2929811 chars from 1137 pages
2026-03-24 20:15:49,010 | INFO |   OS - 3.pdf: 200/


Total PDFs with extracted text : 12  (newly extracted: 12)
  CAO: 1,204,208 chars (merged)
  CN: 1,969,051 chars (merged)
  DSA: 19,385 chars (merged)
  OS: 271,333 chars (merged)


## Cell 8 — Image Extraction from PDFs

In [7]:
SUPPORTED_EXTS = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}


def extract_images_from_pdf(
    pdf_path: Path,
    output_folder: Path,
    min_width: int = MIN_IMAGE_WIDTH,
    min_height: int = MIN_IMAGE_HEIGHT,
) -> int:
    output_folder.mkdir(parents=True, exist_ok=True)
    doc = fitz.open(pdf_path)
    extracted = 0
    for page_num, page in enumerate(doc, start=1):
        for img_idx, img in enumerate(page.get_images(full=True), start=1):
            xref = img[0]
            base = doc.extract_image(xref)
            w, h = base.get("width", 0), base.get("height", 0)
            if w < min_width or h < min_height:
                continue
            ext  = base.get("ext", "png")
            dest = output_folder / f"{pdf_path.stem}_page{page_num}_img{img_idx}.{ext}"
            if dest.exists():
                continue
            dest.write_bytes(base["image"])
            extracted += 1
    doc.close()
    return extracted


total_new_images = 0
for code, pdfs in subject_books.items():
    for pdf in tqdm(pdfs, desc=f"[{code}] Extracting images"):
        try:
            n = extract_images_from_pdf(pdf, IMAGE_DIR)
            total_new_images += n
        except Exception as exc:
            logger.error("Image extraction failed for %s: %s", pdf.name, exc)

all_images = sorted([p for p in IMAGE_DIR.rglob("*") if p.suffix.lower() in SUPPORTED_EXTS])
print(f"\nNew images extracted : {total_new_images}")
print(f"Total images on disk : {len(all_images)}")

[CAO] Extracting images:   0%|          | 0/2 [00:00<?, ?it/s]

[CN] Extracting images:   0%|          | 0/3 [00:00<?, ?it/s]

[DSA] Extracting images:   0%|          | 0/4 [00:00<?, ?it/s]

[OS] Extracting images:   0%|          | 0/3 [00:00<?, ?it/s]


New images extracted : 0
Total images on disk : 969


## Cell 9 — Build Per-Image Metadata with Subject Tag

In [8]:
SUBJECT_CODE_RE = re.compile(r"^([A-Za-z0-9]+)\s*-\s*", re.IGNORECASE)


def infer_subject_from_filename(stem: str) -> str:
    m = SUBJECT_CODE_RE.match(stem)
    return m.group(1).upper() if m else "UNKNOWN"


image_items: List[Dict[str, Any]] = []
for p in all_images:
    rel = p.relative_to(ROOT)
    subject = infer_subject_from_filename(p.stem)
    image_items.append({
        "id": f"img::{rel.as_posix()}",
        "path": p,
        "relative_path": rel.as_posix(),
        "subject": subject,
    })

subject_image_count = defaultdict(int)
for item in image_items:
    subject_image_count[item["subject"]] += 1

print(f"Total image items: {len(image_items)}")
for code, cnt in sorted(subject_image_count.items()):
    print(f"  {code}: {cnt} images")

Total image items: 969
  CAO: 38 images
  CN: 68 images
  DSA: 809 images
  OS: 54 images


## Cell 10 — Vision Model: Image Captioning

In [9]:
CAPTION_CHECKPOINT = CHECKPOINT_DIR / "captions.json"

CAPTION_SYSTEM = (
    "You are a Computer Science technical writer. "
    "Write precise, factual captions for textbook figures."
)
CAPTION_PROMPT_LONG = (
    "Describe the key CS concept, the major components, data flows, and important "
    "relationships shown in this figure in 3-5 sentences using accurate technical terminology."
)
CAPTION_PROMPT_SHORT = (
    "Give a concise 1-2 sentence technical caption of this CS textbook figure."
)


def get_image_mime(path: Path) -> str:
    return {"png": "image/png", "jpg": "image/jpeg", "jpeg": "image/jpeg",
            "webp": "image/webp", "bmp": "image/bmp"}.get(path.suffix.lower().lstrip("."), "image/jpeg")


def prepare_image_b64(path: Path, max_side: int = 1024, quality: int = 85) -> Tuple[str, str]:
    with Image.open(path) as img:
        img = img.convert("RGB")
        if max(img.size) > max_side:
            img.thumbnail((max_side, max_side), Image.Resampling.LANCZOS)
        buf = BytesIO()
        img.save(buf, format="JPEG", quality=quality, optimize=True)
    return base64.b64encode(buf.getvalue()).decode(), "image/jpeg"


def _is_context_limit_error(exc: Exception) -> bool:
    msg = str(exc).lower()
    return "maximum model length" in msg or "decoder prompt" in msg or ("prompt" in msg and "longer than" in msg)


def _vision_call(b64: str, mime: str, user_prompt: str) -> str:
    resp = vision_client.chat.completions.create(
        model=VISION_MODEL,
        max_tokens=512,
        messages=[
            {"role": "system", "content": CAPTION_SYSTEM},
            {"role": "user", "content": [
                {"type": "text",      "text": user_prompt},
                {"type": "image_url", "image_url": {"url": f"data:{mime};base64,{b64}"}},
            ]},
        ],
    )
    return (resp.choices[0].message.content or "").strip()


def caption_image(path: Path) -> str:
    b64, mime = prepare_image_b64(path, max_side=1024)
    try:
        return with_retry(lambda: _vision_call(b64, mime, CAPTION_PROMPT_LONG))
    except Exception as exc:
        if not _is_context_limit_error(exc):
            raise
    logger.warning("Context limit — retrying compact for %s", path.name)
    b64s, mimes = prepare_image_b64(path, max_side=768, quality=75)
    return with_retry(lambda: _vision_call(b64s, mimes, CAPTION_PROMPT_SHORT))


# ---------------------------------------------------------------------------
# Run captioning with checkpoint
# ---------------------------------------------------------------------------
captions_state: Dict[str, Any] = load_json(CAPTION_CHECKPOINT, default={})

pending = [item for item in image_items if item["id"] not in captions_state]
logger.info("%d images to caption (%d already done)", len(pending), len(captions_state))

for item in tqdm(pending, desc="Captioning images"):
    try:
        cap = caption_image(item["path"])
        captions_state[item["id"]] = {
            "caption": cap,
            "relative_path": item["relative_path"],
            "subject": item["subject"],
            "vision_model": VISION_MODEL,
            "captioned_at": datetime.now(timezone.utc).isoformat(),
        }
        dump_json(CAPTION_CHECKPOINT, captions_state)
    except Exception as exc:
        logger.error("Caption failed: %s — %s", item["relative_path"], exc)

print(f"\nTotal captions: {len(captions_state)}")
if captions_state:
    k = next(iter(captions_state))
    print("Sample:", captions_state[k]["caption"][:200])

2026-03-24 20:25:15,211 | INFO | 0 images to caption (969 already done)


Captioning images: 0it [00:00, ?it/s]


Total captions: 969
Sample: This stylized diagram illustrates the architecture of a computer system, showing its core components: a Processor (containing Control and Datapath units), Memory, Input/Output interfaces, and a Compil


## Cell 11 — Vision Model: Image Q&A Generation

For each captioned figure the vision model is asked to generate a **self-contained question-answer pair** directly from the image, producing diagram-grounded training samples.

In [2]:
IMAGE_QA_CHECKPOINT = CHECKPOINT_DIR / "image_qa.json"

IMAGE_QA_SYSTEM = (
    "You are an expert Computer Science professor creating exam questions from textbook figures. "
    "Return ONLY a valid JSON object with keys: question, answer, difficulty (easy/medium/hard), type."
)

IMAGE_QA_PROMPT = """Look at this Computer Science textbook figure carefully.
Generate one insightful exam question that a student can answer only by understanding this figure,
along with a detailed model answer.

Return ONLY a JSON object like:
{"question": "...", "answer": "...", "difficulty": "medium", "type": "diagram"}"""


def generate_image_qa(path: Path) -> Optional[Dict[str, str]]:
    b64, mime = prepare_image_b64(path, max_side=1024)

    def _call():
        resp = vision_client.chat.completions.create(
            model=VISION_MODEL,
            max_tokens=600,
            messages=[
                {"role": "system", "content": IMAGE_QA_SYSTEM},
                {"role": "user", "content": [
                    {"type": "text",      "text": IMAGE_QA_PROMPT},
                    {"type": "image_url", "image_url": {"url": f"data:{mime};base64,{b64}"}},
                ]},
            ],
        )
        return (resp.choices[0].message.content or "").strip()

    raw = with_retry(_call)
    # Parse the JSON object
    cleaned = re.sub(r"```(?:json)?\s*", "", raw).strip().rstrip("`").strip()
    try:
        obj = json.loads(cleaned)
        if isinstance(obj, dict) and "question" in obj and "answer" in obj:
            return obj
    except Exception:
        pass
    m = re.search(r"\{.*\}", cleaned, re.DOTALL)
    if m:
        try:
            return json.loads(m.group(0))
        except Exception:
            pass
    return None


# ---------------------------------------------------------------------------
# Run image Q&A generation  (only for images that have captions)
# ---------------------------------------------------------------------------
image_qa_state: Dict[str, Any] = load_json(IMAGE_QA_CHECKPOINT, default={})

# Filter: only process images whose captions are non-trivial
MIN_CAPTION_LEN = 80
qa_candidates = [
    item for item in image_items
    if item["id"] not in image_qa_state
    and len(captions_state.get(item["id"], {}).get("caption", "")) >= MIN_CAPTION_LEN
]

logger.info("%d images to generate Q&A for (%d already done)", len(qa_candidates), len(image_qa_state))

for item in tqdm(qa_candidates, desc="Generating image Q&A"):
    try:
        qa = generate_image_qa(item["path"])
        if qa:
            image_qa_state[item["id"]] = {
                **qa,
                "image_path": item["relative_path"],
                "subject": item["subject"],
                "caption": captions_state.get(item["id"], {}).get("caption", ""),
                "source": "vision_model",
                "generated_at": datetime.now(timezone.utc).isoformat(),
            }
            dump_json(IMAGE_QA_CHECKPOINT, image_qa_state)
    except Exception as exc:
        logger.error("Image Q&A failed: %s — %s", item["relative_path"], exc)

print(f"\nTotal image Q&A pairs: {len(image_qa_state)}")

NameError: name 'CHECKPOINT_DIR' is not defined

## Cell 12 — Text Embeddings for Captions

In [11]:
EMBED_CHECKPOINT = CHECKPOINT_DIR / "caption_embeddings.json"


def embed_texts(texts: List[str]) -> List[List[float]]:
    def _call():
        resp = embedding_client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
        return [d.embedding for d in resp.data]
    return with_retry(_call)


def embed_query(query: str) -> List[float]:
    return embed_texts([query])[0]


embed_state: Dict[str, Any] = load_json(EMBED_CHECKPOINT, default={})
pending_ids = [k for k in captions_state if k not in embed_state]

for batch_ids in tqdm(list(chunked(pending_ids, EMBED_BATCH)), desc="Embedding captions"):
    batch_texts = [captions_state[_id]["caption"] for _id in batch_ids]
    try:
        vecs = embed_texts(batch_texts)
        for _id, vec in zip(batch_ids, vecs):
            embed_state[_id] = {
                "embedding": vec,
                "embedding_model": EMBEDDING_MODEL,
                "embedded_at": datetime.now(timezone.utc).isoformat(),
            }
        dump_json(EMBED_CHECKPOINT, embed_state)
    except Exception as exc:
        logger.error("Embedding batch failed: %s", exc)

print(f"Caption embeddings in state: {len(embed_state)}")

Embedding captions: 0it [00:00, ?it/s]

Caption embeddings in state: 969


## Cell 13 — Book Text Chunk Embeddings

In [12]:
BOOK_EMBED_CHECKPOINT = CHECKPOINT_DIR / "book_text_embeddings.json"
book_embed_state: Dict[str, Any] = load_json(BOOK_EMBED_CHECKPOINT, default={})

for book_name, cleaned in tqdm(cleaned_texts.items(), desc="Embedding book chunks"):
    if book_name in book_embed_state:
        continue
    chunks = chunk_text(cleaned)
    vectors: List[List[float]] = []
    for batch in chunked(chunks, EMBED_BATCH):
        try:
            vectors.extend(embed_texts(batch))
        except Exception as exc:
            logger.error("Book chunk embed failed (%s): %s", book_name, exc)
            vectors.extend([[] for _ in batch])
    book_embed_state[book_name] = {
        "book": book_name,
        "chunk_texts": chunks,
        "embeddings": vectors,
        "embedding_model": EMBEDDING_MODEL,
        "embedded_at": datetime.now(timezone.utc).isoformat(),
    }
    dump_json(BOOK_EMBED_CHECKPOINT, book_embed_state)

total_chunks = sum(len(v["chunk_texts"]) for v in book_embed_state.values())
print(f"Total book text chunks embedded: {total_chunks}")

Embedding book chunks:   0%|          | 0/12 [00:00<?, ?it/s]

Total book text chunks embedded: 3234


## Cell 14 — ChromaDB Initialisation and Upsert

In [13]:
chroma_client = chromadb.PersistentClient(path=str(VECTORDB_DIR))
COLLECTION_NAME = "cs_textbook_index"
collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"description": "CS textbook captions + text chunks", "hnsw:space": "cosine"},
)

records: List[Dict[str, Any]] = []

# --- Image caption records ---
for _id, cap_obj in captions_state.items():
    emb_obj = embed_state.get(_id)
    if not emb_obj or not emb_obj.get("embedding"):
        continue
    records.append({
        "id": _id,
        "embedding": emb_obj["embedding"],
        "document": cap_obj["caption"],
        "metadata": {
            "source_type": "image_caption",
            "subject": cap_obj.get("subject", "UNKNOWN"),
            "relative_path": cap_obj["relative_path"],
            "vision_model": cap_obj["vision_model"],
            "embedding_model": emb_obj["embedding_model"],
        },
    })

# --- Book text chunk records ---
for book_name, payload in book_embed_state.items():
    subject = infer_subject_from_filename(Path(book_name).stem)
    for idx, (chunk, emb) in enumerate(zip(payload["chunk_texts"], payload["embeddings"])):
        if not emb:
            continue
        records.append({
            "id": f"txt::{book_name}::{idx}",
            "embedding": emb,
            "document": chunk,
            "metadata": {
                "source_type": "book_text_chunk",
                "subject": subject,
                "book": book_name,
                "chunk_index": idx,
                "embedding_model": payload.get("embedding_model", EMBEDDING_MODEL),
            },
        })

# Upsert in batches
for batch in tqdm(list(chunked(records, UPSERT_BATCH)), desc="Upserting to ChromaDB"):
    collection.upsert(
        ids=[r["id"] for r in batch],
        embeddings=[r["embedding"] for r in batch],
        documents=[r["document"] for r in batch],
        metadatas=[r["metadata"] for r in batch],
    )

print(f"ChromaDB total indexed: {collection.count()}")
print(f"  Image caption records : {sum(1 for r in records if r['metadata']['source_type'] == 'image_caption')}")
print(f"  Book text chunk records: {sum(1 for r in records if r['metadata']['source_type'] == 'book_text_chunk')}")

2026-03-24 21:48:38,994 | INFO | Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.


Upserting to ChromaDB:   0%|          | 0/66 [00:00<?, ?it/s]

ChromaDB total indexed: 4203
  Image caption records : 969
  Book text chunk records: 3234


## Cell 15 — RAG Retrieval Helpers

In [1]:
def retrieve_context(
    query: str,
    subject: Optional[str] = None,
    source_type: Optional[str] = None,
    top_k: int = 5,
) -> str:
    """
    Retrieve top-k relevant snippets from ChromaDB, optionally filtering
    by subject and/or source_type (image_caption | book_text_chunk).
    """
    if collection.count() == 0:
        return ""

    vec = embed_query(query)
    where: Dict[str, Any] = {}
    if subject and source_type:
        where = {"$and": [{"subject": subject}, {"source_type": source_type}]}
    elif subject:
        where = {"subject": subject}
    elif source_type:
        where = {"source_type": source_type}

    kwargs: Dict[str, Any] = dict(
        query_embeddings=[vec],
        n_results=min(top_k, collection.count()),
        include=["documents", "metadatas", "distances"],
    )
    if where:
        kwargs["where"] = where

    res = collection.query(**kwargs)
    lines = []
    for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        src = meta.get("relative_path") or meta.get("book", "")
        lines.append(f"[{dist:.3f}] {src}: {doc[:400]}")
    return "\n".join(lines)


# Quick sanity check
test_q = "What is pipelining in CPU?"
ctx = retrieve_context(test_q, subject="CAO", top_k=3)
print(f"Test retrieval for '{test_q}':")
print(ctx[:600] or "(no results — run previous cells first)")

NameError: name 'Optional' is not defined

## Cell 16 — Question-Type Prompt Templates

Each template produces a **different style** of training pair to ensure diverse fine-tuning coverage.

In [15]:
QTYPE_PROMPTS: Dict[str, str] = {

    # ------------------------------------------------------------------
    "conceptual": """\
You are generating conceptual Q&A training data for a {subject_full} tutoring model.
Subject: {subject_full}
Key topics: {key_topics}

Textbook excerpt:
{chunk}

Relevant context:
{context}

Generate exactly {n} JSON objects in a JSON array. Each object must have:
  instruction : a clear, standalone conceptual question (no references to "the text" or "above")
  input       : "" (empty string)
  output      : a thorough, accurate answer (3-6 sentences)
  topic       : the specific sub-topic (e.g. "Cache Coherence", "Paging")
  difficulty  : "easy" | "medium" | "hard"
  type        : "conceptual"
  subject     : "{subject_code}"

Rules: vary difficulty, avoid duplicate questions, no hallucination.
Return ONLY valid JSON array.""",

    # ------------------------------------------------------------------
    "mcq": """\
You are generating multiple-choice questions (MCQ) for a {subject_full} exam.
Subject: {subject_full}

Textbook excerpt:
{chunk}

Relevant context:
{context}

Generate exactly {n} JSON objects in a JSON array. Each object must have:
  instruction : the MCQ stem (a clear question)
  input       : a JSON string containing choices, e.g. '{{"A":"...","B":"...","C":"...","D":"..."}}'
  output      : the correct option letter followed by a 1-2 sentence explanation, e.g. "B — Explanation..."
  topic       : specific sub-topic
  difficulty  : "easy" | "medium" | "hard"
  type        : "mcq"
  subject     : "{subject_code}"

Rules: exactly 4 choices, one clearly correct answer, plausible distractors.
Return ONLY valid JSON array.""",

    # ------------------------------------------------------------------
    "code": """\
You are generating coding/algorithm problems for a {subject_full} programming tutor.
Subject: {subject_full}
Language hint: {code_language}

Textbook excerpt:
{chunk}

Generate exactly {n} JSON objects in a JSON array. Each object must have:
  instruction : a clear coding or algorithm problem statement
  input       : any constraints or example input/output needed (or "" if none)
  output      : a complete, commented, correct code solution or algorithm trace
  topic       : specific sub-topic
  difficulty  : "easy" | "medium" | "hard"
  type        : "code"
  subject     : "{subject_code}"

Rules: solutions must be compilable/runnable, include complexity analysis where applicable.
Return ONLY valid JSON array.""",

    # ------------------------------------------------------------------
    "truefalse": """\
You are generating True/False questions with explanations for a {subject_full} quiz.
Subject: {subject_full}

Textbook excerpt:
{chunk}

Generate exactly {n} JSON objects in a JSON array. Each object must have:
  instruction : a declarative statement that is definitively True or False
  input       : "" (empty)
  output      : "True" or "False" followed by a 2-3 sentence explanation of why
  topic       : specific sub-topic
  difficulty  : "easy" | "medium" | "hard"
  type        : "truefalse"
  subject     : "{subject_code}"

Rules: mix True and False statements (roughly 50/50), make plausible false statements.
Return ONLY valid JSON array.""",

    # ------------------------------------------------------------------
    "diagram": """\
You are generating diagram-based questions for a {subject_full} visual learning module.
Subject: {subject_full}

Figure caption / description:
{context}

Textbook excerpt:
{chunk}

Generate exactly {n} JSON objects in a JSON array. Each object must have:
  instruction : a question whose answer requires understanding the described figure/diagram
  input       : brief description of the figure the student is looking at
  output      : a detailed answer referencing specific components in the figure
  topic       : specific sub-topic
  difficulty  : "easy" | "medium" | "hard"
  type        : "diagram"
  subject     : "{subject_code}"

Rules: questions must be answerable from the figure description alone.
Return ONLY valid JSON array.""",

    # ------------------------------------------------------------------
    "scenario": """\
You are generating scenario-based application questions for a {subject_full} problem-solving tutor.
Subject: {subject_full}

Textbook excerpt:
{chunk}

Relevant context:
{context}

Generate exactly {n} JSON objects in a JSON array. Each object must have:
  instruction : a real-world scenario or system design problem that requires applying {subject_full} concepts
  input       : specific parameters/constraints of the scenario
  output      : a structured, step-by-step solution with reasoning
  topic       : specific sub-topic
  difficulty  : "medium" | "hard"
  type        : "scenario"
  subject     : "{subject_code}"

Rules: scenarios must be realistic, outputs must show clear reasoning, not just results.
Return ONLY valid JSON array.""",
}


def build_prompt(
    qtype: str,
    subject_code: str,
    chunk: str,
    context: str,
    n: int = PAIRS_PER_CHUNK,
) -> str:
    meta = get_subject_meta(subject_code)
    template = QTYPE_PROMPTS[qtype]
    return template.format(
        subject_code=subject_code,
        subject_full=meta["full_name"],
        key_topics=", ".join(meta["key_topics"][:8]),
        code_language=meta.get("code_language", "Python / C"),
        chunk=chunk,
        context=context,
        n=n,
    )


print("Prompt templates loaded for types:", list(QTYPE_PROMPTS.keys()))

Prompt templates loaded for types: ['conceptual', 'mcq', 'code', 'truefalse', 'diagram', 'scenario']


## Cell 17 — Core Synthetic Pair Generator

In [16]:
def generate_pairs(
    qtype: str,
    subject_code: str,
    chunk: str,
    context: str,
    book_name: str,
    n: int = PAIRS_PER_CHUNK,
) -> List[Dict[str, Any]]:
    """
    Call the LLM to generate `n` Q&A pairs of `qtype` from `chunk`.
    Returns a list of validated, normalised dicts.
    """
    prompt = build_prompt(qtype, subject_code, chunk, context, n)

    def _call():
        resp = text_client.chat.completions.create(
            model=SYNTHETIC_MODEL,
            temperature=0.75,
            max_tokens=2048,
            messages=[
                {"role": "system", "content": "You produce strict JSON arrays as instructed. No preamble, no markdown."},
                {"role": "user",   "content": prompt},
            ],
        )
        return (resp.choices[0].message.content or "[]").strip()

    raw = with_retry(_call)
    pairs = extract_json_array(raw)

    validated = []
    for p in pairs:
        if not isinstance(p, dict):
            continue
        if not p.get("instruction") or not p.get("output"):
            continue
        # Normalise / inject mandatory fields
        p.setdefault("input",      "")
        p.setdefault("topic",      subject_code)
        p.setdefault("type",       qtype)
        p.setdefault("subject",    subject_code)
        p.setdefault("difficulty", "medium")
        p["book"]   = book_name
        p["source"] = "synthetic"
        p["hash"]   = stable_hash(p["instruction"] + p["output"])
        validated.append(p)

    return validated


print("Core generator defined.")

Core generator defined.


## Cell 18 — Per-Subject Synthetic Data Generation (Main Loop)

This is the main generation loop. For each subject:
1. Split the merged book text into chunks.
2. For each question type, retrieve relevant context and generate pairs.
3. Save per-subject JSONL files with incremental checkpointing.
4. Deduplicate by hash.

In [ ]:
SYNTH_CHECKPOINT_DIR = CHECKPOINT_DIR / "synthetic"
SYNTH_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)


def load_subject_checkpoint(subject_code: str) -> Dict[str, Any]:
    path = SYNTH_CHECKPOINT_DIR / f"{subject_code}_state.json"
    return load_json(path, default={"done_keys": [], "pairs": []})


def save_subject_checkpoint(subject_code: str, state: Dict[str, Any]) -> None:
    path = SYNTH_CHECKPOINT_DIR / f"{subject_code}_state.json"
    dump_json(path, state)


def run_subject(subject_code: str, subject_pdfs: List[Path]) -> List[Dict[str, Any]]:
    """
    Full generation run for one subject. Resumes from checkpoint if interrupted.
    Returns list of all generated pairs.
    """
    meta = get_subject_meta(subject_code)
    qtype_weights = meta["qtype_weights"]
    total_weight   = sum(qtype_weights.values())
    text = subject_text.get(subject_code, "")

    if not text:
        logger.warning("[%s] No text found — skipping", subject_code)
        return []

    # Allocate chunks per question type proportionally to weights
    all_chunks = chunk_text(text, CHUNK_CHARS, CHUNK_OVERLAP)
    # Use up to MAX_CHUNKS_PER_BOOK * number of books
    budget = MAX_CHUNKS_PER_BOOK * len(subject_pdfs)
    all_chunks = all_chunks[:budget]

    qtype_chunk_alloc: Dict[str, List[str]] = defaultdict(list)
    for i, chunk in enumerate(all_chunks):
        # Assign chunk to question type round-robin weighted by weight
        # Build a sorted list of types weighted by their share
        idx_in_weight = i % total_weight
        cumulative = 0
        for qtype, w in qtype_weights.items():
            cumulative += w
            if idx_in_weight < cumulative:
                qtype_chunk_alloc[qtype].append(chunk)
                break

    state = load_subject_checkpoint(subject_code)
    done_keys = set(state["done_keys"])
    all_pairs = state["pairs"]
    seen_hashes = {p["hash"] for p in all_pairs}

    logger.info(
        "[%s] %s — %d chunks across %d qtypes, %d already done",
        subject_code, meta["full_name"], len(all_chunks), len(qtype_chunk_alloc), len(done_keys),
    )

    for qtype, chunks in qtype_chunk_alloc.items():
        for chunk_idx, chunk in enumerate(tqdm(chunks, desc=f"[{subject_code}][{qtype}]", leave=False)):
            job_key = f"{subject_code}::{qtype}::{chunk_idx}"
            if job_key in done_keys:
                continue

            # Retrieve context: image captions for diagram type, text chunks otherwise
            ctx_source = "image_caption" if qtype == "diagram" else "book_text_chunk"
            context = retrieve_context(
                query=chunk[:300],
                subject=subject_code,
                source_type=ctx_source,
                top_k=4,
            )

            try:
                book_name = subject_pdfs[chunk_idx % len(subject_pdfs)].name
                pairs = generate_pairs(
                    qtype=qtype,
                    subject_code=subject_code,
                    chunk=chunk,
                    context=context,
                    book_name=book_name,
                    n=PAIRS_PER_CHUNK,
                )
                new_count = 0
                for p in pairs:
                    h = p["hash"]
                    if h not in seen_hashes:
                        all_pairs.append(p)
                        seen_hashes.add(h)
                        new_count += 1

                done_keys.add(job_key)
                state["done_keys"] = list(done_keys)
                state["pairs"]     = all_pairs
                save_subject_checkpoint(subject_code, state)

            except Exception as exc:
                logger.error("[%s][%s] chunk %d failed: %s", subject_code, qtype, chunk_idx, exc)

    logger.info("[%s] Done — %d pairs", subject_code, len(all_pairs))
    return all_pairs


# ---------------------------------------------------------------------------
# RUN all subjects
# ---------------------------------------------------------------------------
all_subject_pairs: Dict[str, List[Dict[str, Any]]] = {}

for code, pdfs in subject_books.items():
    print(f"\n{'='*60}")
    print(f"  Processing: {code} — {get_subject_meta(code)['full_name']}")
    print(f"{'='*60}")
    pairs = run_subject(code, pdfs)
    all_subject_pairs[code] = pairs
    print(f"  Generated {len(pairs)} pairs for {code}")

total_pairs = sum(len(v) for v in all_subject_pairs.values())
print(f"\nTotal pairs across all subjects: {total_pairs}")

## Cell 19 — Integrate Vision-Generated Image Q&A Pairs

In [ ]:
def convert_image_qa_to_pair(img_id: str, qa_obj: Dict[str, Any]) -> Dict[str, Any]:
    """Convert image_qa_state entry into the standard training pair format."""
    return {
        "instruction": qa_obj.get("question", ""),
        "input":       f"[Figure: {qa_obj.get('image_path', '')}] {qa_obj.get('caption', '')[:300]}",
        "output":      qa_obj.get("answer", ""),
        "topic":       qa_obj.get("subject", "UNKNOWN"),
        "difficulty":  qa_obj.get("difficulty", "medium"),
        "type":        "diagram",
        "subject":     qa_obj.get("subject", "UNKNOWN"),
        "book":        Path(qa_obj.get("image_path", "")).stem.split("_page")[0],
        "source":      "vision_model",
        "hash":        stable_hash(qa_obj.get("question", "") + qa_obj.get("answer", "")),
    }


vision_pairs_by_subject: Dict[str, List[Dict]] = defaultdict(list)
for img_id, qa_obj in image_qa_state.items():
    if not qa_obj.get("question") or not qa_obj.get("answer"):
        continue
    pair   = convert_image_qa_to_pair(img_id, qa_obj)
    subj   = qa_obj.get("subject", "UNKNOWN")
    vision_pairs_by_subject[subj].append(pair)

# Merge vision pairs into subject pair lists
for subj, vpairs in vision_pairs_by_subject.items():
    existing_hashes = {p["hash"] for p in all_subject_pairs.get(subj, [])}
    new_vision = [p for p in vpairs if p["hash"] not in existing_hashes]
    all_subject_pairs.setdefault(subj, []).extend(new_vision)
    if new_vision:
        print(f"  [{subj}] +{len(new_vision)} vision-generated diagram pairs")

total_after_vision = sum(len(v) for v in all_subject_pairs.values())
print(f"\nTotal pairs after adding vision Q&A: {total_after_vision}")

## Cell 20 — Generate Router Training Dataset

Creates training data for the **prompt router** that decides which subject adapter to activate. Each sample is a user query labelled with the correct subject.

In [ ]:
ROUTER_CHECKPOINT = CHECKPOINT_DIR / "router_state.json"
ROUTER_SAMPLES_PER_SUBJECT = 50

ROUTER_PROMPT = """\
You are generating routing training data for a subject classifier that routes
student questions to the correct CS subject expert.

Subject: {subject_full} (code: {subject_code})
Description: {description}
Key topics: {key_topics}

Generate exactly {n} JSON objects in a JSON array. Each object must have:
  query   : a realistic student question that clearly belongs to this subject
  subject : "{subject_code}"
  label   : "{subject_full}"

Rules:
- Vary the question style (conceptual, problem-solving, debugging, definition)
- Include short questions and long questions
- Some questions should use jargon, some should use plain language
- NO question should be ambiguous between subjects
Return ONLY a valid JSON array.
"""


def generate_router_samples(subject_code: str, n: int = ROUTER_SAMPLES_PER_SUBJECT) -> List[Dict]:
    meta = get_subject_meta(subject_code)
    prompt = ROUTER_PROMPT.format(
        subject_code=subject_code,
        subject_full=meta["full_name"],
        description=meta["description"],
        key_topics=", ".join(meta["key_topics"]),
        n=n,
    )

    def _call():
        resp = text_client.chat.completions.create(
            model=SYNTHETIC_MODEL,
            temperature=0.8,
            max_tokens=2048,
            messages=[
                {"role": "system", "content": "Return only valid JSON arrays."},
                {"role": "user",   "content": prompt},
            ],
        )
        return (resp.choices[0].message.content or "[]").strip()

    raw = with_retry(_call)
    samples = extract_json_array(raw)
    valid = []
    for s in samples:
        if isinstance(s, dict) and s.get("query") and s.get("subject"):
            s["hash"] = stable_hash(s["query"])
            valid.append(s)
    return valid


router_state: Dict[str, Any] = load_json(ROUTER_CHECKPOINT, default={"done": [], "samples": []})
done_subjects = set(router_state["done"])
router_samples = router_state["samples"]

for code in subject_books:
    if code in done_subjects:
        print(f"  [{code}] router samples already generated — skipping")
        continue
    print(f"  [{code}] generating {ROUTER_SAMPLES_PER_SUBJECT} router samples…")
    try:
        samples = generate_router_samples(code, ROUTER_SAMPLES_PER_SUBJECT)
        router_samples.extend(samples)
        done_subjects.add(code)
        router_state["done"]    = list(done_subjects)
        router_state["samples"] = router_samples
        dump_json(ROUTER_CHECKPOINT, router_state)
        print(f"     → {len(samples)} samples added")
    except Exception as exc:
        logger.error("Router generation failed for %s: %s", code, exc)

print(f"\nTotal router training samples: {len(router_samples)}")

## Cell 21 — Save Outputs (Per-Subject + Combined + Router)

In [ ]:
def write_jsonl(path: Path, records: List[Dict]) -> int:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in records:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    return len(records)


# --- Per-subject files ---
per_subject_stats: Dict[str, Dict[str, Any]] = {}

for code, pairs in all_subject_pairs.items():
    out_path = SUBJECT_OUT / f"{code}_finetune.jsonl"
    n = write_jsonl(out_path, pairs)

    type_counts  = defaultdict(int)
    diff_counts  = defaultdict(int)
    src_counts   = defaultdict(int)
    for p in pairs:
        type_counts[p.get("type", "?")]       += 1
        diff_counts[p.get("difficulty", "?")]  += 1
        src_counts[p.get("source", "?")]       += 1

    per_subject_stats[code] = {
        "total": n,
        "file":  str(out_path),
        "by_type":       dict(type_counts),
        "by_difficulty": dict(diff_counts),
        "by_source":     dict(src_counts),
    }
    print(f"  [{code}] {n} pairs → {out_path.name}")

# --- Combined file ---
all_combined = [p for pairs in all_subject_pairs.values() for p in pairs]
combined_path = OUTPUT_DIR / "combined_finetune.jsonl"
write_jsonl(combined_path, all_combined)
print(f"\nCombined: {len(all_combined)} pairs → {combined_path.name}")

# --- Router training file ---
router_path = OUTPUT_DIR / "router_training.jsonl"
write_jsonl(router_path, router_samples)
print(f"Router  : {len(router_samples)} samples → {router_path.name}")

## Cell 22 — Pipeline Statistics Report

In [ ]:
stats = {
    "run_timestamp":    datetime.now(timezone.utc).isoformat(),
    "vision_model":     VISION_MODEL,
    "synthetic_model":  SYNTHETIC_MODEL,
    "embedding_model":  EMBEDDING_MODEL,
    "subjects_processed": list(subject_books.keys()),
    "total_pdfs":         sum(len(v) for v in subject_books.values()),
    "total_images":       len(image_items),
    "total_captions":     len(captions_state),
    "total_image_qa":     len(image_qa_state),
    "total_text_chunks":  total_chunks,
    "vectordb_records":   collection.count(),
    "total_synth_pairs":  len(all_combined),
    "router_samples":     len(router_samples),
    "per_subject":        per_subject_stats,
    "output_files": {
        "combined":       str(combined_path),
        "router":         str(router_path),
        "per_subject_dir": str(SUBJECT_OUT),
    },
}

stats_path = OUTPUT_DIR / "pipeline_stats.json"
dump_json(stats_path, stats)

# Pretty print summary
print("\n" + "=" * 62)
print("  PIPELINE SUMMARY")
print("=" * 62)
print(f"  Total PDFs processed    : {stats['total_pdfs']}")
print(f"  Total images extracted  : {stats['total_images']}")
print(f"  Captions generated      : {stats['total_captions']}")
print(f"  Image Q&A pairs (vision): {stats['total_image_qa']}")
print(f"  Book text chunks        : {stats['total_text_chunks']}")
print(f"  ChromaDB records        : {stats['vectordb_records']}")
print(f"  Synthetic pairs (total) : {stats['total_synth_pairs']}")
print(f"  Router training samples : {stats['router_samples']}")
print()
print(f"  {'Subject':<8} {'Total':>7}  {'by type'}")
print("  " + "-" * 58)
for code, s in per_subject_stats.items():
    type_str = ", ".join(f"{t}:{c}" for t, c in sorted(s["by_type"].items()))
    print(f"  {code:<8} {s['total']:>7}  {type_str}")
print("=" * 62)
print(f"\nStats saved to: {stats_path}")

## Cell 23 — (Optional) Export to HuggingFace Datasets Format

Converts the JSONL files to HuggingFace `datasets` format for direct use with `Trainer` or `trl.SFTTrainer`.

In [ ]:
try:
    from datasets import Dataset, DatasetDict
    HF_AVAILABLE = True
except ImportError:
    HF_AVAILABLE = False
    print("datasets library not installed — skipping HF export. Run: pip install datasets")

if HF_AVAILABLE:
    HF_OUT = OUTPUT_DIR / "hf_datasets"
    HF_OUT.mkdir(parents=True, exist_ok=True)

    def build_alpaca_text(row: Dict[str, Any]) -> str:
        """Format each sample in Alpaca instruction style."""
        if row.get("input"):
            return (
                f"### Instruction:\n{row['instruction']}\n\n"
                f"### Input:\n{row['input']}\n\n"
                f"### Response:\n{row['output']}"
            )
        return (
            f"### Instruction:\n{row['instruction']}\n\n"
            f"### Response:\n{row['output']}"
        )

    # Per-subject datasets
    subject_datasets: Dict[str, Dataset] = {}
    for code, pairs in all_subject_pairs.items():
        if not pairs:
            continue
        for p in pairs:
            p["text"] = build_alpaca_text(p)
        ds = Dataset.from_list(pairs)
        ds.save_to_disk(str(HF_OUT / code))
        subject_datasets[code] = ds
        print(f"  [{code}] HF dataset saved → {HF_OUT / code}  ({len(ds)} rows)")

    # Combined DatasetDict (train/test split 95/5)
    for p in all_combined:
        p["text"] = build_alpaca_text(p)
    combined_ds = Dataset.from_list(all_combined)
    split = combined_ds.train_test_split(test_size=0.05, seed=42)
    split.save_to_disk(str(HF_OUT / "combined_split"))
    print(f"\nCombined split saved → {HF_OUT / 'combined_split'}")
    print(f"  train: {len(split['train'])}  test: {len(split['test'])}")

    # Router dataset
    router_ds = Dataset.from_list(router_samples)
    router_split = router_ds.train_test_split(test_size=0.1, seed=42)
    router_split.save_to_disk(str(HF_OUT / "router_split"))
    print(f"Router split saved  → {HF_OUT / 'router_split'}")
    print(f"  train: {len(router_split['train'])}  test: {len(router_split['test'])}")

## Cell 24 — Quality Validation Samples

Print a representative sample of each question type across all subjects to spot-check quality.

In [ ]:
QTYPES_TO_SHOW = ["conceptual", "mcq", "code", "truefalse", "diagram", "scenario"]

for code in all_subject_pairs:
    print(f"\n{'━'*60}")
    print(f"  {code} — {get_subject_meta(code)['full_name']}")
    print(f"{'━'*60}")

    pairs_by_type: Dict[str, List] = defaultdict(list)
    for p in all_subject_pairs[code]:
        pairs_by_type[p.get("type", "?")].append(p)

    for qtype in QTYPES_TO_SHOW:
        pool = pairs_by_type.get(qtype, [])
        if not pool:
            continue
        sample = pool[0]  # show first sample
        print(f"\n  [{qtype.upper()}] topic={sample.get('topic')}  difficulty={sample.get('difficulty')}")
        print(f"  Q: {sample['instruction'][:200]}")
        if sample.get("input"):
            print(f"  I: {str(sample['input'])[:150]}")
        print(f"  A: {sample['output'][:300]}")

## Cell 25 — VectorDB Similarity Search Validation

In [ ]:
TEST_QUERIES = {
    "CAO": "How does pipelining improve CPU performance?",
    "CN":  "What is the difference between TCP and UDP?",
    "DSA": "Explain the time complexity of quicksort.",
    "OS":  "How does the OS handle a page fault?",
}

for code, query in TEST_QUERIES.items():
    if code not in subject_books:
        continue
    print(f"\n[{code}] Query: {query}")
    ctx = retrieve_context(query, subject=code, top_k=3)
    for line in ctx.split("\n")[:3]:
        print(f"  {line[:150]}")

## Cell 26 — Pipeline Status Checkpoint (Resume-Safe)

In [ ]:
PIPELINE_STATUS = CHECKPOINT_DIR / "pipeline_status.json"

final_status = {
    "stage":             "complete",
    "completed_at":      datetime.now(timezone.utc).isoformat(),
    "captions":          len(captions_state),
    "caption_embeddings": len(embed_state),
    "image_qa_pairs":    len(image_qa_state),
    "book_embeddings":   len(book_embed_state),
    "vectordb_indexed":  collection.count(),
    "synth_pairs_total": len(all_combined),
    "router_samples":    len(router_samples),
    "checkpoints": {
        "captions":         str(CAPTION_CHECKPOINT),
        "caption_embeddings": str(EMBED_CHECKPOINT),
        "image_qa":         str(IMAGE_QA_CHECKPOINT),
        "book_embeddings":  str(BOOK_EMBED_CHECKPOINT),
        "router":           str(ROUTER_CHECKPOINT),
        "synthetic_dir":    str(SYNTH_CHECKPOINT_DIR),
    },
}

dump_json(PIPELINE_STATUS, final_status)
logger.info("Pipeline complete. Status written to %s", PIPELINE_STATUS)

print("\n✅ All done!")
print(f"   Outputs  : {OUTPUT_DIR}")
print(f"   Subjects : {SUBJECT_OUT}")
print(f"   Status   : {PIPELINE_STATUS}")